# Proyek Akhir Kriptografi — Kelompok 7 (Ilmu Komputer UNNES 2026)
## Konstruksi S-box AES Berbasis *Surrogate-Assisted Genetic Algorithm* (SAGA) Melintasi 30 Polinomial Irreducible $\text{GF}(2^8)$ untuk Enkripsi Citra

| No | Nama Anggota | NIM | Bagian Tugas di Notebook |
|---|---|---|---|
| 1 | **Cahya Din Ramadhan** | `2404130018` | **Bagian 1:** Eksplorasi 30 polinomial irreducible $\text{GF}(2^8)$, dataset matriks, & ablasi 4 model ML Surrogate (KNN, RF, Ridge 1-Baris vs Ridge Pairwise-XOR) |
| 2 | **Muhammad Nabil Fikri** | `2404130167` | **Bagian 2:** Optimasi multi-objektif *Surrogate-Assisted Genetic Algorithm* (SAGA), optimasi konstanta $C$, konstruksi `S-box K7`, & uji dasar (Balance, Bijective, NL, SAC) |
| 3 | **Fathan Nabil Rahman** | `2404130140` | **Bagian 3:** Pengujian lanjutan S-box (BIC-NL, BIC-SAC, LAP, DAP, Fixed Point, Opposite Fixed Point, Siklus) & komparasi vs AES, Alamsyah (2023), dan Alamsyah (2025) `S-box₄₄` |
| 4 | **Rashya Farellyndo Revanza** | `2404130132` | **Bagian 4:** Implementasi AES-128 T-table (FIPS-197), mode CBC dua arah dengan *domain-separated IV*, enkripsi citra Mandrill, Histogram ($\chi^2$), Entropy, Korelasi H/V/D, NPCR, & UACI |

**Referensi Utama:**
1. Daemen, J., & Rijmen, V. (2002). *The Design of Rijndael* (AES / FIPS-197). Springer.
2. Wu, Y., Noonan, J. P., & Agaian, S. (2011). *NPCR and UACI randomness tests for image encryption*. Cyber Journals: JSAT 1(2).
3. Alamsyah, Prasetiyo, B., & Muhammad, A. (2023). *S-box Construction on AES Algorithm using Affine Matrix Modification to Improve Image Encryption Security*. Scientific Journal of Informatics 10(2).
4. Alamsyah, Setiawan, A., Putra, A. T., Budiman, K., Muslim, M. A., Salahudin, S. N., & Prasetiyo, B. (2025). *AES S-box modification uses affine matrices exploration for increased S-box strength*. Nonlinear Dynamics 113:3869–3890.

> **Catatan Eksekusi:** Notebook ini dirancang **berurutan penuh (*Run All safe*)**. Klik **Runtime → Run all** di Google Colab; seluruh bagian akan berjalan tanpa ketergantungan mundur.

---
# Bagian 1 — Eksplorasi 30 Polinomial $\text{GF}(2^8)$, Dataset, & Pemodelan ML Surrogate
**Penanggung jawab:** Cahya Din Ramadhan (`2404130018`)

Menindaklanjuti saran *future research* Alamsyah dkk. (2025, hal. 3888), konstruksi dimulai dengan mengevaluasi seluruh **30 polinomial irreducible derajat 8** di $\text{GF}(2)[x]$ — tidak hanya `0x11B` (AES / Alamsyah 2025) atau `0x1F3` (Alamsyah 2023) — lalu membangun model *Machine Learning* untuk memprediksi kualitas **SAC** sekaligus **BIC-SAC** dari matriks affine non-sirkulan di $GL(8,2)$.

In [ ]:
import json
import os
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsRegressor

np.set_printoptions(linewidth=150)
SEED = 7

# Konstanta utilitas bit (urutan b0..b7, b0 = LSB)
BITS = ((np.arange(256)[:, None] >> np.arange(8)) & 1).astype(np.int64)
POW2 = 1 << np.arange(8)
X_ARR = np.arange(256, dtype=np.int64)
PAIR_I, PAIR_J = np.triu_indices(8, k=1)  # 28 pasangan bit output (j < k)
C_AES_INT = 0x63

print("Pustaka siap. SEED =", SEED)

## 1.1 Memuat Citra Uji Mandrill (512×512 RGB)
Jika `mandrill.png` belum diunggah ke direktori Colab, sel berikut otomatis mengunduh citra standar Baboon/Mandrill ($512 \times 512$ RGB).

In [ ]:
IMG_PATH = "mandrill.png"
if not os.path.exists(IMG_PATH):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/mikolalysenko/baboon-image/master/baboon.png",
        IMG_PATH,
    )

img = np.array(Image.open(IMG_PATH).convert("RGB"))
print("Ukuran citra uji:", img.shape, "| tipe:", img.dtype)

plt.figure(figsize=(3.4, 3.4))
plt.imshow(img)
plt.axis("off")
plt.title("Citra Uji Mandrill (512×512 RGB)")
plt.tight_layout()
plt.show()

## 1.2 Aritmetika $\text{GF}(2^8)$, Validasi Referensi, & Skrining Seluruh 30 Polinomial Irreducible
Di $\text{GF}(2)[x]$ terdapat tepat **30 polinomial irreducible derajat 8**. Kita verifikasi dulu implementasi aritmetika $\text{GF}(2^8)$ dengan mereproduksi persis:
1. **AES standar** (`0x11B`, FIPS-197)
2. **Alamsyah dkk. (2023)** — `S-box1 (A0)`, `S-box2 (A2)`, `S-box3 (A1)` pada polinomial `0x1F3`
3. **Alamsyah dkk. (2025)** — `S-box₄₄ (K44)` pada polinomial `0x11B` (Tabel 5 & Tabel 16 paper)

Kemudian seluruh 30 polinomial irreducible dievaluasi secara objektif untuk memilih polinomial terbaik.

In [ ]:
def poly_mod(a: int, b: int) -> int:
    db = b.bit_length() - 1
    while a.bit_length() - 1 >= db:
        a ^= b << (a.bit_length() - 1 - db)
    return a


def get_irreducible_polynomials():
    return [
        p for p in range(0x101, 0x200, 2)
        if all(poly_mod(p, d) != 0 for d in range(2, 32))
    ]


def poly_to_str(poly: int) -> str:
    terms = []
    for deg in range(8, -1, -1):
        if (poly >> deg) & 1:
            terms.append("1" if deg == 0 else ("x" if deg == 1 else f"x^{deg}"))
    return " + ".join(terms)


def gf_mul(a: int, b: int, poly: int) -> int:
    r = 0
    while b:
        if b & 1:
            r ^= a
        a <<= 1
        if a & 0x100:
            a ^= poly
        b >>= 1
    return r


def gf_inverse_table(poly: int) -> np.ndarray:
    inv = np.zeros(256, dtype=np.int64)
    for a in range(1, 256):
        base, exp, res = a, 254, 1
        while exp:
            if exp & 1:
                res = gf_mul(res, base, poly)
            base = gf_mul(base, base, poly)
            exp >>= 1
        inv[a] = res
    return inv


_INV_CACHE = {}


def get_inv(poly: int) -> np.ndarray:
    if poly not in _INV_CACHE:
        _INV_CACHE[poly] = gf_inverse_table(poly)
    return _INV_CACHE[poly]


def gf2_rank_int(rows_int) -> int:
    basis = []
    for v in rows_int:
        x = int(v)
        for b in basis:
            if (x ^ b) < x:
                x ^= b
        if x:
            basis.append(x)
            basis.sort(reverse=True)
    return len(basis)


def build_sbox(A, poly: int, c_int: int = C_AES_INT) -> np.ndarray:
    inv = get_inv(poly)
    y = (BITS[inv] @ (np.asarray(A, dtype=np.int64) % 2).T) % 2
    return ((y @ POW2).astype(np.int64)) ^ int(c_int)


def build_sbox_from_rows(rows_int, poly: int, c_int: int = C_AES_INT) -> np.ndarray:
    return build_sbox(BITS[np.asarray(rows_int, dtype=np.int64)], poly=poly, c_int=c_int)


# --- Matriks acuan untuk validasi ---
A_AES = [[1 if (c - r) % 8 in (0, 4, 5, 6, 7) else 0 for c in range(8)] for r in range(8)]
A0_2023 = [[1,1,0,0,0,0,0,1],[1,1,1,0,0,0,0,0],[0,1,1,1,0,0,0,0],[0,0,1,1,1,0,0,0],
           [0,0,0,1,1,1,0,0],[0,0,0,0,1,1,1,0],[0,0,0,0,0,1,1,1],[1,0,0,0,0,0,1,1]]
A1_2023 = [[0,1,1,1,1,0,1,0],[0,0,1,1,1,1,0,1],[1,0,0,1,1,1,1,0],[0,1,0,0,1,1,1,1],
           [1,0,1,0,0,1,1,1],[1,1,0,1,0,0,1,1],[1,1,1,0,1,0,0,1],[1,1,1,1,0,1,0,0]]
A2_2023 = [[1,1,1,0,1,0,0,1],[1,1,1,1,0,1,0,0],[0,1,1,1,1,0,1,0],[0,0,1,1,1,1,0,1],
           [1,0,0,1,1,1,1,0],[0,1,0,0,1,1,1,1],[1,0,1,0,0,1,1,1],[1,1,0,1,0,0,1,1]]
K44_2025 = [[0,1,0,1,0,1,1,1],[1,0,1,0,1,0,1,1],[1,1,0,1,0,1,0,1],[1,1,1,0,1,0,1,0],
            [0,1,1,1,0,1,0,1],[1,0,1,1,1,0,1,0],[0,1,0,1,1,1,0,1],[1,0,1,0,1,1,1,0]]

S_AES = build_sbox(A_AES, poly=0x11B)
S_ALAMSYAH_2025 = build_sbox(K44_2025, poly=0x11B)
S_A0_2023 = build_sbox(A0_2023, poly=0x1F3)
S_A2_2023 = build_sbox(A2_2023, poly=0x1F3)
S_A1_2023 = build_sbox(A1_2023, poly=0x1F3)

assert list(S_AES[:4]) == [0x63, 0x7C, 0x77, 0x7B], "AES standar tidak cocok!"
assert list(S_ALAMSYAH_2025[:16]) == [99, 205, 85, 71, 25, 127, 113, 219, 63, 244, 109, 159, 11, 228, 94, 214], "S-box44 (2025) tidak cocok!"
assert list(S_A0_2023[:4]) == [99, 224, 149, 199], "S-box1 A0 (2023) tidak cocok!"
assert np.array_equal(np.array(A2_2023), np.roll(np.array(A1_2023), 2, axis=0))
print("✔ Validasi AES standar (0x11B), Alamsyah 2023 (0x1F3), & Alamsyah 2025 S-box44 (0x11B) berhasil persis!")
print("  Catatan: A2_2023 adalah rotasi siklis 2 baris dari A1_2023 (sehingga metrik SAC & BIC keduanya identik).")

# --- Skrining seluruh 30 polinomial irreducible ---
def precompute_component_sac(poly: int):
    inv = get_inv(poly)
    row_avg = np.zeros(256, dtype=np.float64)
    row_dev = np.zeros(256, dtype=np.float64)
    for r in range(1, 256):
        f = (BITS[inv] @ BITS[r]) % 2
        probs = np.array([(f ^ f[X_ARR ^ (1 << i)]).mean() for i in range(8)])
        row_avg[r] = probs.mean()
        row_dev[r] = np.abs(probs - 0.5).mean()
    return row_avg, row_dev, np.abs(row_avg - 0.5)


irreds = get_irreducible_polynomials()
assert len(irreds) == 30
poly_rows = []
for p in irreds:
    r_avg, r_dev, r_bdev = precompute_component_sac(p)
    t16 = float(np.sort(r_dev[1:])[:16].mean())
    t32 = float(np.sort(r_bdev[1:])[:32].mean())
    poly_rows.append({
        "poly_int": p,
        "Polinomial (Hex)": f"0x{p:03X}",
        "Bentuk Polinomial": poly_to_str(p),
        "Top-16 Dev SAC": round(t16, 5),
        "Top-32 Dev BIC-SAC": round(t32, 5),
        "Skor Pool Gabungan": round(t16 + t32, 5),
    })

df_poly = pd.DataFrame(poly_rows).sort_values(["Skor Pool Gabungan", "Top-16 Dev SAC", "poly_int"]).reset_index(drop=True)
POLY_K7 = int(df_poly.loc[0, "poly_int"])
print(f"\nTotal polinomial irreducible derajat 8: {len(df_poly)}")
print(f"Polinomial peringkat #1 terpilih: 0x{POLY_K7:03X} ({poly_to_str(POLY_K7)})")
display(df_poly.drop(columns=["poly_int"]).head(10))

## 1.3 Ekstraksi Dataset Matriks Affine Invertible dari Bit-Plane Citra Mandrill
Bit-plane 0 dan 1 dari citra Mandrill disusun menjadi potongan 64-bit (kandidat matriks biner $8 \times 8$), lalu disaring yang memiliki **rank 8 di $\text{GF}(2)$** (*invertible*, sehingga menjamin S-box *balanced* dan *bijective*).

Perhatikan sifat aljabar penting pada $S(x) = A \cdot x^{-1} \oplus C$:
- Bit keluaran ke-$j$ ditentukan oleh baris $r_j$ matriks $A$, sehingga **SAC** bergantung pada 8 baris $r_j$.
- Pasangan XOR bit keluaran $f_j \oplus f_k$ ditentukan oleh vektor $r_j \oplus r_k$, sehingga **BIC-SAC** bergantung pada **28 pasangan XOR** $r_j \oplus r_k$!

In [ ]:
bitstream = np.concatenate([(img >> p) & 1 for p in (0, 1)]).reshape(-1)
n_bits = (len(bitstream) // 64) * 64
kandidat = bitstream[:n_bits].reshape(-1, 8, 8).astype(np.int64)
rows_all = (kandidat * POW2).sum(2)
dataset_rows = np.array([r for r in rows_all if gf2_rank_int(r) == 8], dtype=np.int64)

r_avg_k7, r_dev_k7, r_bdev_k7 = precompute_component_sac(POLY_K7)
pairs_all = dataset_rows[:, PAIR_I] ^ dataset_rows[:, PAIR_J]

y_sac_dev = r_dev_k7[dataset_rows].mean(1)
y_sac_avg = r_avg_k7[dataset_rows].mean(1)
y_bic_dev = r_bdev_k7[pairs_all].mean(1)
y_bic_avg = r_avg_k7[pairs_all].mean(1)
y_joint = 1.0 * y_sac_dev + 1.5 * y_bic_dev + 5.0 * (np.abs(y_sac_avg - 0.5) + np.abs(y_bic_avg - 0.5))

print(f"Total kandidat 8×8 dari citra : {len(kandidat):,}")
print(f"Matriks invertible (rank = 8) : {len(dataset_rows):,} ({len(dataset_rows)/len(kandidat)*100:.2f}%)")
print(f"Rentang Deviasi SAC dataset   : {y_sac_dev.min():.5f} – {y_sac_dev.max():.5f}")
print(f"Rentang Deviasi BIC-SAC       : {y_bic_dev.min():.5f} – {y_bic_dev.max():.5f}")

## 1.4 Pelatihan & Studi Ablasi 4 Model Machine Learning (Split 80:20)
Kita membandingkan 4 model pada data uji 20% (tanpa *data leakage*):
1. **KNN Regressor (64-bit mentah)** — *baseline* sesuai contoh mata kuliah.
2. **Random Forest Regressor (64-bit mentah)** — *baseline* pohon keputusan.
3. **Ridge Regression 1-Baris (*bag-of-rows*, 256-dim)** — berhasil memprediksi SAC, namun **gagal memprediksi BIC-SAC**.
4. **Ridge Komposit Usulan (*Row + Pairwise-XOR*, 256+256-dim)** — memprediksi **SAC dan BIC-SAC secara akurat**, lalu bobotnya dipakai sebagai *surrogate model* super-cepat di Algoritma Genetik (Bagian 2).

In [ ]:
def build_features(rows_mat: np.ndarray):
    n = len(rows_mat)
    idx = np.arange(n)
    X_row = np.zeros((n, 256), dtype=np.float64)
    for k in range(8):
        X_row[idx, rows_mat[:, k]] += 1.0
    pairs = rows_mat[:, PAIR_I] ^ rows_mat[:, PAIR_J]
    X_pair = np.zeros((n, 256), dtype=np.float64)
    for k in range(28):
        X_pair[idx, pairs[:, k]] += 1.0
    return X_row, X_pair


X_raw = BITS[dataset_rows].reshape(len(dataset_rows), 64).astype(np.float64)
X_row, X_pair = build_features(dataset_rows)

rng_split = np.random.default_rng(SEED)
perm = rng_split.permutation(len(dataset_rows))
n_te = int(0.2 * len(dataset_rows))
idx_te, idx_tr = perm[:n_te], perm[n_te:]

# 1. Baseline KNN (64-bit mentah)
knn_sac = KNeighborsRegressor(n_neighbors=15, n_jobs=-1).fit(X_raw[idx_tr], y_sac_dev[idx_tr])
knn_bic = KNeighborsRegressor(n_neighbors=15, n_jobs=-1).fit(X_raw[idx_tr], y_bic_dev[idx_tr])

# 2. Baseline Random Forest (64-bit mentah)
rf_sac = RandomForestRegressor(n_estimators=120, max_depth=14, random_state=SEED, n_jobs=-1).fit(X_raw[idx_tr], y_sac_dev[idx_tr])
rf_bic = RandomForestRegressor(n_estimators=120, max_depth=14, random_state=SEED, n_jobs=-1).fit(X_raw[idx_tr], y_bic_dev[idx_tr])

# 3. Ridge 1-baris (bag-of-rows saja)
m_sdev = Ridge(alpha=1e-3, fit_intercept=False).fit(X_row[idx_tr], y_sac_dev[idx_tr])
m_savg = Ridge(alpha=1e-3, fit_intercept=False).fit(X_row[idx_tr], y_sac_avg[idx_tr])
m_bdev_row = Ridge(alpha=1e-3, fit_intercept=False).fit(X_row[idx_tr], y_bic_dev[idx_tr])

# 4. Ridge Komposit Usulan (Row + Pairwise-XOR)
m_bdev = Ridge(alpha=1e-3, fit_intercept=False).fit(X_pair[idx_tr], y_bic_dev[idx_tr])
m_bavg = Ridge(alpha=1e-3, fit_intercept=False).fit(X_pair[idx_tr], y_bic_avg[idx_tr])

p_knn_sac, p_knn_bic = knn_sac.predict(X_raw[idx_te]), knn_bic.predict(X_raw[idx_te])
p_rf_sac, p_rf_bic = rf_sac.predict(X_raw[idx_te]), rf_bic.predict(X_raw[idx_te])
p_r1_sac, p_r1_bic = m_sdev.predict(X_row[idx_te]), m_bdev_row.predict(X_row[idx_te])
p_rp_bic, p_rp_bavg = m_bdev.predict(X_pair[idx_te]), m_bavg.predict(X_pair[idx_te])

df_ablation = pd.DataFrame([
    ["KNN (64-bit mentah)", "Deviasi SAC", r2_score(y_sac_dev[idx_te], p_knn_sac), mean_absolute_error(y_sac_dev[idx_te], p_knn_sac)],
    ["KNN (64-bit mentah)", "Deviasi BIC-SAC", r2_score(y_bic_dev[idx_te], p_knn_bic), mean_absolute_error(y_bic_dev[idx_te], p_knn_bic)],
    ["Random Forest (64-bit mentah)", "Deviasi SAC", r2_score(y_sac_dev[idx_te], p_rf_sac), mean_absolute_error(y_sac_dev[idx_te], p_rf_sac)],
    ["Random Forest (64-bit mentah)", "Deviasi BIC-SAC", r2_score(y_bic_dev[idx_te], p_rf_bic), mean_absolute_error(y_bic_dev[idx_te], p_rf_bic)],
    ["Ridge 1-Baris (bag-of-rows)", "Deviasi SAC", r2_score(y_sac_dev[idx_te], p_r1_sac), mean_absolute_error(y_sac_dev[idx_te], p_r1_sac)],
    ["Ridge 1-Baris (bag-of-rows)", "Deviasi BIC-SAC", r2_score(y_bic_dev[idx_te], p_r1_bic), mean_absolute_error(y_bic_dev[idx_te], p_r1_bic)],
    ["Ridge Komposit Usulan (Row + Pairwise-XOR)", "Deviasi SAC", r2_score(y_sac_dev[idx_te], p_r1_sac), mean_absolute_error(y_sac_dev[idx_te], p_r1_sac)],
    ["Ridge Komposit Usulan (Row + Pairwise-XOR)", "Deviasi BIC-SAC", r2_score(y_bic_dev[idx_te], p_rp_bic), mean_absolute_error(y_bic_dev[idx_te], p_rp_bic)],
    ["Ridge Komposit Usulan (Row + Pairwise-XOR)", "Rata-rata BIC-SAC", r2_score(y_bic_avg[idx_te], p_rp_bavg), mean_absolute_error(y_bic_avg[idx_te], p_rp_bavg)],
], columns=["Model ML", "Target Prediksi", "R² (Data Uji)", "MAE (Data Uji)"])
display(df_ablation.round(6))

fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))
lim_b = [float(y_bic_dev[idx_te].min()), float(y_bic_dev[idx_te].max())]
for ax, pred, judul, col in [
    (axes[0], p_knn_bic, "KNN (64-bit) → Dev BIC-SAC", "#64748b"),
    (axes[1], p_r1_bic, "Ridge 1-Baris → Dev BIC-SAC (Gagal)", "#dc2626"),
    (axes[2], p_rp_bic, "Ridge Pairwise-XOR → Dev BIC-SAC (Akurat)", "#0f766e"),
]:
    ax.scatter(y_bic_dev[idx_te], pred, s=7, alpha=0.45, color=col)
    ax.plot(lim_b, lim_b, "k--", lw=1)
    ax.set_title(judul, fontsize=9.5)
    ax.set_xlabel("Deviasi BIC-SAC aktual")
    ax.set_ylabel("Prediksi model")
plt.tight_layout()
plt.savefig("ablasi_ml_kelompok7.png", dpi=150)
plt.show()

---
# Bagian 2 — *Surrogate-Assisted Genetic Algorithm* (SAGA), Optimasi Konstanta $C$, & Uji Dasar S-box
**Penanggung jawab:** Muhammad Nabil Fikri (`2404130167`)

Meskipun model ML Surrogate dapat mengevaluasi satu kandidat matriks secara instan, memilih 8 baris bebas linear yang **secara simultan** meminimalkan deviasi SAC (8 baris $r_j$) **dan** deviasi BIC-SAC (28 pasangan XOR $r_j \oplus r_k$) merupakan masalah optimasi kombinatorial kuadratik di ruang $|GL(8,2)| \approx 5{,}35 \times 10^{18}$. Oleh karena itu, **Algoritma Genetik** digunakan untuk mencari matriks affine optimal $A_{\text{K7}}$ dengan dipandu bobot ML Surrogate.

In [ ]:
# Bobot surrogate hasil pelatihan ML Bagian 1 (pembulatan 4 desimal agar deterministik lintas mesin)
ws_d = np.round(m_sdev.coef_ * 8.0, 4)
ws_a = np.round(m_savg.coef_ * 8.0, 4)
wb_d = np.round(m_bdev.coef_ * 28.0, 4)
wb_a = np.round(m_bavg.coef_ * 28.0, 4)

row_ids = np.arange(1, 256, dtype=np.int64)
top_rows = row_ids[np.lexsort((row_ids, ws_d[1:]))]


def surrogate_fitness(ind: np.ndarray) -> float:
    p_xor = ind[PAIR_I] ^ ind[PAIR_J]
    sd = float(ws_d[ind].mean())
    sa = float(ws_a[ind].mean())
    bd = float(wb_d[p_xor].mean())
    ba = float(wb_a[p_xor].mean())
    return round(1.0 * sd + 1.5 * bd + 5.0 * (abs(sa - 0.5) + abs(ba - 0.5)), 5)


POP_SIZE, GENERATIONS = 140, 160
rng_ga = np.random.default_rng(SEED)


def random_valid() -> np.ndarray:
    while True:
        pool = top_rows[:80] if rng_ga.random() < 0.75 else row_ids
        cand = rng_ga.choice(pool, size=8, replace=False)
        if gf2_rank_int(cand) == 8:
            return np.sort(cand)


t0 = time.perf_counter()
pop = [random_valid() for _ in range(POP_SIZE)]
fits = np.array([surrogate_fitness(ind) for ind in pop], dtype=np.float64)
hist_best, hist_mean = [float(fits.min())], [float(fits.mean())]
elite_n = max(4, POP_SIZE // 12)

for gen in range(GENERATIONS):
    order = np.argsort(fits, kind="mergesort")
    new_pop = [pop[i].copy() for i in order[:elite_n]]
    while len(new_pop) < POP_SIZE:
        i1, i2 = rng_ga.integers(0, POP_SIZE, size=2)
        p1 = pop[i1] if fits[i1] <= fits[i2] else pop[i2]
        j1, j2 = rng_ga.integers(0, POP_SIZE, size=2)
        p2 = pop[j1] if fits[j1] <= fits[j2] else pop[j2]
        pool = np.unique(np.concatenate([p1, p2]))
        child = None
        for _ in range(15):
            cand = rng_ga.choice(pool, size=8, replace=False) if len(pool) >= 8 else p1.copy()
            if rng_ga.random() < 0.70:
                for _m in range(1 if rng_ga.random() < 0.75 else 2):
                    pos = int(rng_ga.integers(0, 8))
                    src = top_rows[:50] if rng_ga.random() < 0.70 else row_ids
                    v = int(rng_ga.choice(src))
                    if v not in cand:
                        cand[pos] = v
            if gf2_rank_int(cand) == 8:
                child = np.sort(cand)
                break
        new_pop.append(p1.copy() if child is None else child)
    pop = new_pop
    fits = np.array([surrogate_fitness(ind) for ind in pop], dtype=np.float64)
    hist_best.append(float(fits.min()))
    hist_mean.append(float(fits.mean()))

t_ga = time.perf_counter() - t0
ROWS_K7 = pop[int(np.argmin(fits))].tolist()
A_K7 = BITS[ROWS_K7]
S_BEST_DATASET = build_sbox_from_rows(dataset_rows[int(np.argmin(y_joint))], poly=POLY_K7, c_int=C_AES_INT)

print(f"SAGA selesai dalam {t_ga:.2f} detik ({POP_SIZE} individu × {GENERATIONS} generasi)")
print("Baris matriks affine terpilih:", ROWS_K7)
print("Matriks affine A_K7 (8×8, rank GF(2) =", gf2_rank_int(ROWS_K7), "):")
print(A_K7)

plt.figure(figsize=(8.5, 3.3))
plt.plot(hist_best, label="Best Fitness (Elit SAGA)", color="#0f766e", lw=2)
plt.plot(hist_mean, label="Rata-rata Populasi", color="#64748b", ls="--", lw=1.3)
plt.axhline(y_joint.min(), color="#dc2626", ls=":", lw=1.5, label=f"Terbaik Dataset tanpa SAGA ({y_joint.min():.5f})")
plt.xlabel("Generasi"); plt.ylabel("Skor Rugi Multi-Objektif")
plt.title(f"Konvergensi Surrogate-Assisted Genetic Algorithm (Polinomial 0x{POLY_K7:03X})")
plt.legend(fontsize=8.5); plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig("konvergensi_saga_kelompok7.png", dpi=150)
plt.show()

## 2.1 Optimasi Konstanta 8-bit $C \in \{0, \dots, 255\}$ & Konstruksi Final `S-box K7`
Secara matematis, mengganti konstanta $C$ pada $S(x) = A \cdot x^{-1} \oplus C$ menjaga seluruh nilai **NL, SAC, BIC-NL, BIC-SAC, LAP, dan DAP** tetap **100% invarian**, namun mengubah **Fixed Point** ($S(x)=x$), **Opposite Fixed Point** ($S(x)=255-x$), dan **struktur siklus**. Kita evaluasi seluruh 256 nilai $C$ untuk memperoleh `0 FP`, `0 OFP`, dan **1 siklus tunggal penuh `[256]`**.

In [ ]:
def cycle_lengths(S):
    S = [int(v) for v in S]
    seen, lens = set(), []
    for i in range(256):
        if i not in seen:
            c, cur = 0, i
            while cur not in seen:
                seen.add(cur); cur = S[cur]; c += 1
            lens.append(c)
    return sorted(lens, reverse=True)


base_sbox = build_sbox_from_rows(ROWS_K7, poly=POLY_K7, c_int=0)
best_c, n_zero_fp_ofp, n_single_cycle = None, 0, 0
for c in range(256):
    cand_s = base_sbox ^ c
    fp = int(np.sum(cand_s == X_ARR))
    ofp = int(np.sum(cand_s == (255 - X_ARR)))
    cl = cycle_lengths(cand_s)
    if fp == 0 and ofp == 0:
        n_zero_fp_ofp += 1
        if len(cl) == 1:
            n_single_cycle += 1
    key = (fp + ofp, fp, len(cl), -cl[0], -min(cl), c)
    if best_c is None or key < best_c[0]:
        best_c = (key, c, fp, ofp, cl, cand_s)

C_K7_INT = int(best_c[1])
S_K7 = best_c[5]
INV_S_K7 = np.argsort(S_K7)

print(f"Dari 256 kandidat konstanta C: {n_zero_fp_ofp} bebas FP & OFP, {n_single_cycle} bersiklus tunggal [256].")
print(f"Konstanta C terpilih: 0x{C_K7_INT:02X} ({C_K7_INT}) → FP={best_c[2]}, OFP={best_c[3]}, Siklus={best_c[4]}")

df_sbox_k7 = pd.DataFrame(S_K7.reshape(16, 16), index=[f"0x{i:X}_" for i in range(16)], columns=[f"_0x{j:X}" for j in range(16)])
print("\nTabel S-box K7 (16×16):")
display(df_sbox_k7)

df_sbox_k7.to_csv("sbox_k7.csv")
with open("sbox_k7.txt", "w") as f:
    f.write("S_BOX = [\n" + ",\n".join("    " + ", ".join(f"{int(v):3d}" for v in S_K7[i*16:(i+1)*16]) for i in range(16)) + "\n]\n")
    f.write("\nINV_S_BOX = " + str([int(v) for v in INV_S_K7]) + "\n")
print("✔ Disimpan: sbox_k7.csv dan sbox_k7.txt")

## 2.2 Pengujian Dasar S-box: Balance, Bijectivity, Nonlinearity (NL), & SAC

In [ ]:
def balance(S):
    S = np.asarray(S, dtype=np.int64)
    satu = [int(((S >> j) & 1).sum()) for j in range(8)]
    return {"jumlah_1": satu, "jumlah_0": [256 - v for v in satu], "seimbang": all(v == 128 for v in satu)}


def bijective(S):
    S = np.asarray(S, dtype=np.int64)
    return len(set(int(v) for v in S)) == 256 and int(S.min()) == 0 and int(S.max()) == 255


def walsh(f):
    w = 1 - 2 * np.asarray(f, dtype=np.int64)
    h = 1
    while h < 256:
        w = w.reshape(-1, 2 * h)
        a, b = w[:, :h].copy(), w[:, h:].copy()
        w[:, :h], w[:, h:] = a + b, a - b
        w = w.reshape(-1)
        h *= 2
    return w


def nl_bool(f):
    return int(128 - np.abs(walsh(f)).max() // 2)


def nonlinearity(S):
    S = np.asarray(S, dtype=np.int64)
    per_bit = [nl_bool((S >> j) & 1) for j in range(8)]
    comp = [nl_bool(BITS[S & v].sum(1) % 2) for v in range(1, 256)]
    return {"per_bit": per_bit, "avg": float(np.mean(per_bit)), "min": min(per_bit), "max": max(per_bit), "min_komponen": min(comp)}


def sac(S):
    S = np.asarray(S, dtype=np.int64)
    M = np.array([BITS[S ^ S[X_ARR ^ (1 << i)]].mean(0) for i in range(8)], dtype=np.float64)
    return {"matrix": M, "avg": float(M.mean()), "min": float(M.min()), "max": float(M.max()), "deviasi": float(np.abs(M - 0.5).mean())}


bal_k7 = balance(S_K7)
bij_k7 = bijective(S_K7)
nl_k7 = nonlinearity(S_K7)
sac_k7 = sac(S_K7)

print("Balance      :", "YA (128/128 di semua 8 bit)" if bal_k7["seimbang"] else "TIDAK")
print("Bijective    :", "YA (256 nilai unik 0..255)" if bij_k7 else "TIDAK")
print("Nonlinearity :", nl_k7["per_bit"], f"→ avg {nl_k7['avg']}, min 255 komponen = {nl_k7['min_komponen']}")
print(f"SAC          : rata-rata {sac_k7['avg']:.5f} | deviasi {sac_k7['deviasi']:.5f} | rentang {sac_k7['min']:.4f}–{sac_k7['max']:.4f}")

plt.figure(figsize=(5.0, 4.2))
plt.imshow(sac_k7["matrix"], cmap="RdBu_r", vmin=0.42, vmax=0.58)
for i in range(8):
    for j in range(8):
        plt.text(j, i, f"{sac_k7['matrix'][i, j]:.3f}", ha="center", va="center", fontsize=7)
plt.colorbar(label="Peluang bit output berubah")
plt.xlabel("Bit output j"); plt.ylabel("Bit input i")
plt.title("Matriks SAC — S-box K7 (Kelompok 7)")
plt.tight_layout(); plt.savefig("sac_matrix_kelompok7.png", dpi=150); plt.show()

---
# Bagian 3 — Pengujian Lanjutan S-box (BIC, LAP, DAP, Siklus) & Komparasi Referensi
**Penanggung jawab:** Fathan Nabil Rahman (`2404130140`)

Bagian ini menguji **BIC-NL**, **BIC-SAC**, **LAP** (bias & probabilitas), **DAP**, **Fixed Point**, **Opposite Fixed Point**, serta **Cycle Structure**, lalu membandingkan `S-box K7` secara langsung dengan **AES Standar**, **Alamsyah dkk. (2023)**, dan **`S-box₄₄` Alamsyah dkk. (2025)** termasuk metrik gabungan **$S$ (Pers. 9)** dan **$SV$ (Pers. 20)**.

In [ ]:
def bic(S):
    S = np.asarray(S, dtype=np.int64)
    nls, sacs = [], []
    nl_mat = np.full((8, 8), np.nan)
    sac_mat = np.full((8, 8), np.nan)
    for j in range(8):
        for k in range(j + 1, 8):
            f = ((S >> j) ^ (S >> k)) & 1
            nl = nl_bool(f)
            s = float(np.mean([(f ^ f[X_ARR ^ (1 << i)]).mean() for i in range(8)]))
            nls.append(nl); sacs.append(s)
            nl_mat[j, k] = nl_mat[k, j] = nl
            sac_mat[j, k] = sac_mat[k, j] = s
    arr = np.array(sacs)
    return {
        "nl_min": min(nls), "nl_avg": float(np.mean(nls)),
        "sac_avg": float(arr.mean()), "sac_deviasi": float(np.abs(arr - 0.5).mean()),
        "sac_min": float(arr.min()), "sac_max": float(arr.max()),
        "nl_matrix": nl_mat, "sac_matrix": sac_mat,
    }


def lap_detail(S):
    S = np.asarray(S, dtype=np.int64)
    lat = np.zeros((256, 256), dtype=np.int16)
    lat[0, 0] = 256
    for b in range(1, 256):
        g = BITS[S & b].sum(1) % 2
        lat[:, b] = walsh(g)
    sub = np.abs(lat[1:, 1:])
    idx = np.unravel_index(np.argmax(sub), sub.shape)
    max_corr = int(sub[idx])
    bias = max_corr / 512.0
    return {"lat": lat, "max_corr": max_corr, "bias": float(bias), "prob": float(0.5 + bias),
            "mask_a": int(idx[0] + 1), "mask_b": int(idx[1] + 1)}


def dap_detail(S):
    S = np.asarray(S, dtype=np.int64)
    ddt = np.zeros((256, 256), dtype=np.uint16)
    for dx in range(1, 256):
        ddt[dx] = np.bincount(S ^ S[X_ARR ^ dx], minlength=256)
    pos = np.unravel_index(np.argmax(ddt[1:]), ddt[1:].shape)
    dx_max, dy_max = int(pos[0] + 1), int(pos[1])
    mc = int(ddt[dx_max, dy_max])
    return {"ddt": ddt, "max_count": mc, "dx": dx_max, "dy": dy_max, "dap": float(mc / 256.0)}


bic_k7 = bic(S_K7)
lap_k7 = lap_detail(S_K7)
dap_k7 = dap_detail(S_K7)

print(f"BIC-NL   : min = {bic_k7['nl_min']} | avg = {bic_k7['nl_avg']:.2f}")
print(f"BIC-SAC  : avg = {bic_k7['sac_avg']:.5f} | deviasi = {bic_k7['sac_deviasi']:.5f} | rentang {bic_k7['sac_min']:.4f}–{bic_k7['sac_max']:.4f}")
print(f"LAP      : bias maks = {lap_k7['bias']:.6f} (prob maks = {lap_k7['prob']:.4f}, korelasi = {lap_k7['max_corr']} di a={lap_k7['mask_a']}, b={lap_k7['mask_b']})")
print(f"DAP      : {dap_k7['dap']:.6f} ({dap_k7['max_count']}/256 di Δx={dap_k7['dx']}, Δy={dap_k7['dy']})")

fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.2))
im0 = axes[0, 0].imshow(bic_k7["nl_matrix"], cmap="RdBu_r", vmin=104, vmax=112)
axes[0, 0].set_title("Matriks BIC-NL — S-box K7"); fig.colorbar(im0, ax=axes[0, 0], fraction=0.046)
for i in range(8):
    for j in range(8):
        if not np.isnan(bic_k7["nl_matrix"][i, j]):
            axes[0, 0].text(j, i, f"{bic_k7['nl_matrix'][i, j]:.0f}", ha="center", va="center", fontsize=7)

im1 = axes[0, 1].imshow(bic_k7["sac_matrix"], cmap="RdBu_r", vmin=0.46, vmax=0.54)
axes[0, 1].set_title("Matriks BIC-SAC — S-box K7"); fig.colorbar(im1, ax=axes[0, 1], fraction=0.046)
for i in range(8):
    for j in range(8):
        if not np.isnan(bic_k7["sac_matrix"][i, j]):
            axes[0, 1].text(j, i, f"{bic_k7['sac_matrix'][i, j]:.3f}", ha="center", va="center", fontsize=6.5)

im2 = axes[1, 0].imshow(lap_k7["lat"][1:, 1:], aspect="auto", cmap="RdBu_r", vmin=-32, vmax=32)
axes[1, 0].set_title("Linear Approximation Table (LAT)"); axes[1, 0].set_xlabel("Mask b"); axes[1, 0].set_ylabel("Mask a")
fig.colorbar(im2, ax=axes[1, 0], fraction=0.046)

im3 = axes[1, 1].imshow(dap_k7["ddt"][1:], aspect="auto", cmap="viridis")
axes[1, 1].set_title("Differential Distribution Table (DDT)"); axes[1, 1].set_xlabel("Δy"); axes[1, 1].set_ylabel("Δx")
fig.colorbar(im3, ax=axes[1, 1], fraction=0.046)

plt.tight_layout(); plt.savefig("uji_lanjutan_sbox_kelompok7.png", dpi=150); plt.show()

## 3.1 Perbandingan Komprehensif: `S-box K7` vs AES Standar, Alamsyah (2023), & Alamsyah (2025) `S-box₄₄`

In [ ]:
def ringkas_metrik(nama, S):
    bal = balance(S)["seimbang"]
    bij = bijective(S)
    nl = nonlinearity(S)
    sc = sac(S)
    bc = bic(S)
    lp = lap_detail(S)["bias"]
    dp = dap_detail(S)["dap"]
    fp = int(np.sum(S == X_ARR))
    ofp = int(np.sum(S == (255 - X_ARR)))
    cl = cycle_lengths(S)
    s_2025 = 0.5 * (abs(sc["avg"] - 0.5) + abs(bc["sac_avg"] - 0.5))
    sv_2025 = (120.0 - nl["avg"]) + abs(0.5 - sc["avg"]) + (120.0 - bc["nl_avg"]) + abs(0.5 - bc["sac_avg"])
    return {
        "S-box": nama,
        "Bal/Bij": "Ya/Ya" if (bal and bij) else "Tidak",
        "NL": nl["min"],
        "SAC": round(sc["avg"], 5),
        "Dev SAC": round(sc["deviasi"], 5),
        "BIC-NL": bc["nl_min"],
        "BIC-SAC": round(bc["sac_avg"], 5),
        "Dev BIC-SAC": round(bc["sac_deviasi"], 5),
        "LAP (bias)": round(lp, 4),
        "DAP": round(dp, 6),
        "Skor S (2025)": round(s_2025, 5),
        "Skor SV (2025)": round(sv_2025, 5),
        "FP / OFP": f"{fp} / {ofp}",
        "Siklus": f"{len(cl)} ({cl[0]})",
    }


daftar_sbox = [
    ("S-box K7 (Kelompok 7 · 0x17B)", S_K7),
    ("AES Standar (FIPS-197 · 0x11B)", S_AES),
    ("Alamsyah 2025 S-box44 (0x11B)", S_ALAMSYAH_2025),
    ("Alamsyah 2023 A0 (0x1F3)", S_A0_2023),
    ("Alamsyah 2023 A2 (0x1F3)", S_A2_2023),
    ("Alamsyah 2023 A1 (0x1F3)", S_A1_2023),
    ("Terbaik Dataset tanpa SAGA", S_BEST_DATASET),
]

df_banding = pd.DataFrame([ringkas_metrik(nm, S) for nm, S in daftar_sbox])
display(df_banding)
df_banding.to_csv("perbandingan_sbox_kelompok7.csv", index=False)

# Simpan metadata lengkap untuk integrasi / dokumentasi
with open("sbox_k7_metadata.json", "w") as f:
    json.dump({
        "kelompok": "Kelompok 7 - Ilmu Komputer UNNES 2026",
        "polynomial_hex": f"0x{POLY_K7:03X}",
        "polynomial": poly_to_str(POLY_K7),
        "affine_rows": ROWS_K7,
        "constant_c_hex": f"0x{C_K7_INT:02X}",
        "metrics": df_banding.iloc[0].to_dict(),
        "sbox": [int(v) for v in S_K7],
    }, f, indent=2)

fig, axes = plt.subplots(1, 3, figsize=(13.0, 3.8))
lbls = [n.split(" (")[0] for n in df_banding["S-box"]]
cols = ["#0f766e" if "K7" in n else "#94a3b8" for n in df_banding["S-box"]]
for ax, kol, ttl in [
    (axes[0], "Dev SAC", "Deviasi SAC (makin kecil makin baik)"),
    (axes[1], "Dev BIC-SAC", "Deviasi BIC-SAC (makin kecil makin baik)"),
    (axes[2], "Skor S (2025)", "Skor S Alamsyah 2025 (ideal = 0)"),
]:
    b = ax.bar(range(len(lbls)), df_banding[kol], color=cols)
    ax.set_xticks(range(len(lbls))); ax.set_xticklabels(lbls, rotation=22, ha="right", fontsize=8)
    ax.set_title(ttl, fontsize=9.5); ax.bar_label(b, fmt="%.4f", fontsize=7.5)
plt.tight_layout(); plt.savefig("perbandingan_sbox_kelompok7.png", dpi=150); plt.show()

---
# Bagian 4 — Implementasi AES-128 (T-table, FIPS-197) & Evaluasi Enkripsi Citra
**Penanggung jawab:** Rashya Farellyndo Revanza (`2404130132`)

`S-box K7` dipasang langsung menggantikan S-box standar pada tahapan **SubBytes / InvSubBytes** dan **Key Expansion (`SubWord`)** di AES-128, sementara **MixColumns / InvMixColumns** tetap menggunakan $\text{GF}(2^8)$ AES standar (`0x11B`).
Mode operasi menggunakan **CBC Dua Arah dengan *domain-separated IV*** (`IV_rev = IV XOR 0xA5`) agar perubahan 1 piksel merambat ke seluruh citra sekaligus mencegah penggunaan ulang IV identik antar-lintasan.

In [ ]:
RCON = [0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80, 0x1B, 0x36]
CHI2_CRIT = 293.2478
NPCR_IDEAL, UACI_IDEAL = (1.0 - 2.0 ** -8) * 100.0, 33.4635


def _xtime(a: int) -> int:
    a <<= 1
    return (a ^ 0x11B) & 0xFF if a & 0x100 else a


def _gf_mul_aes(a: int, b: int) -> int:
    r = 0
    while b:
        if b & 1:
            r ^= a
        a = _xtime(a); b >>= 1
    return r


def _rotr32(w: int, n: int) -> int:
    return ((w >> n) | (w << (32 - n))) & 0xFFFFFFFF


class AES128:
    def __init__(self, key: bytes, sbox):
        assert len(key) == 16
        S = [int(v) for v in sbox]
        assert sorted(S) == list(range(256)), "S-box wajib bijektif"
        Si = [0] * 256
        for i, v in enumerate(S):
            Si[v] = i
        self.S, self.Si = S, Si
        te0 = [(_gf_mul_aes(s, 2) << 24) | (s << 16) | (s << 8) | _gf_mul_aes(s, 3) for s in S]
        td0 = [(_gf_mul_aes(s, 14) << 24) | (_gf_mul_aes(s, 9) << 16) | (_gf_mul_aes(s, 13) << 8) | _gf_mul_aes(s, 11) for s in Si]
        self.Te = [te0, [_rotr32(w, 8) for w in te0], [_rotr32(w, 16) for w in te0], [_rotr32(w, 24) for w in te0]]
        self.Td = [td0, [_rotr32(w, 8) for w in td0], [_rotr32(w, 16) for w in td0], [_rotr32(w, 24) for w in td0]]
        w = [int.from_bytes(key[4*i:4*i+4], "big") for i in range(4)]
        for i in range(4, 44):
            t = w[i - 1]
            if i % 4 == 0:
                t = ((t << 8) | (t >> 24)) & 0xFFFFFFFF
                t = (S[t >> 24] << 24) | (S[(t >> 16) & 255] << 16) | (S[(t >> 8) & 255] << 8) | S[t & 255]
                t ^= RCON[i // 4 - 1] << 24
            w.append(w[i - 4] ^ t)
        self.ek = [w[4*r:4*r+4] for r in range(11)]
        Td = self.Td
        inv_mix = lambda x: Td[0][S[x >> 24]] ^ Td[1][S[(x >> 16) & 255]] ^ Td[2][S[(x >> 8) & 255]] ^ Td[3][S[x & 255]]
        self.dk = [self.ek[10]] + [[inv_mix(x) for x in self.ek[10 - r]] for r in range(1, 10)] + [self.ek[0]]

    def encrypt_words(self, s0, s1, s2, s3):
        T0, T1, T2, T3 = self.Te; k = self.ek
        s0 ^= k[0][0]; s1 ^= k[0][1]; s2 ^= k[0][2]; s3 ^= k[0][3]
        for r in range(1, 10):
            kr = k[r]
            s0, s1, s2, s3 = (
                T0[s0 >> 24] ^ T1[(s1 >> 16) & 255] ^ T2[(s2 >> 8) & 255] ^ T3[s3 & 255] ^ kr[0],
                T0[s1 >> 24] ^ T1[(s2 >> 16) & 255] ^ T2[(s3 >> 8) & 255] ^ T3[s0 & 255] ^ kr[1],
                T0[s2 >> 24] ^ T1[(s3 >> 16) & 255] ^ T2[(s0 >> 8) & 255] ^ T3[s1 & 255] ^ kr[2],
                T0[s3 >> 24] ^ T1[(s0 >> 16) & 255] ^ T2[(s1 >> 8) & 255] ^ T3[s2 & 255] ^ kr[3],
            )
        S, kr = self.S, k[10]
        return (
            ((S[s0 >> 24] << 24) | (S[(s1 >> 16) & 255] << 16) | (S[(s2 >> 8) & 255] << 8) | S[s3 & 255]) ^ kr[0],
            ((S[s1 >> 24] << 24) | (S[(s2 >> 16) & 255] << 16) | (S[(s3 >> 8) & 255] << 8) | S[s0 & 255]) ^ kr[1],
            ((S[s2 >> 24] << 24) | (S[(s3 >> 16) & 255] << 16) | (S[(s0 >> 8) & 255] << 8) | S[s1 & 255]) ^ kr[2],
            ((S[s3 >> 24] << 24) | (S[(s0 >> 16) & 255] << 16) | (S[(s1 >> 8) & 255] << 8) | S[s2 & 255]) ^ kr[3],
        )

    def decrypt_words(self, s0, s1, s2, s3):
        T0, T1, T2, T3 = self.Td; k = self.dk
        s0 ^= k[0][0]; s1 ^= k[0][1]; s2 ^= k[0][2]; s3 ^= k[0][3]
        for r in range(1, 10):
            kr = k[r]
            s0, s1, s2, s3 = (
                T0[s0 >> 24] ^ T1[(s3 >> 16) & 255] ^ T2[(s2 >> 8) & 255] ^ T3[s1 & 255] ^ kr[0],
                T0[s1 >> 24] ^ T1[(s0 >> 16) & 255] ^ T2[(s3 >> 8) & 255] ^ T3[s2 & 255] ^ kr[1],
                T0[s2 >> 24] ^ T1[(s1 >> 16) & 255] ^ T2[(s0 >> 8) & 255] ^ T3[s3 & 255] ^ kr[2],
                T0[s3 >> 24] ^ T1[(s2 >> 16) & 255] ^ T2[(s1 >> 8) & 255] ^ T3[s0 & 255] ^ kr[3],
            )
        S, kr = self.Si, k[10]
        return (
            ((S[s0 >> 24] << 24) | (S[(s3 >> 16) & 255] << 16) | (S[(s2 >> 8) & 255] << 8) | S[s1 & 255]) ^ kr[0],
            ((S[s1 >> 24] << 24) | (S[(s0 >> 16) & 255] << 16) | (S[(s3 >> 8) & 255] << 8) | S[s2 & 255]) ^ kr[1],
            ((S[s2 >> 24] << 24) | (S[(s1 >> 16) & 255] << 16) | (S[(s0 >> 8) & 255] << 8) | S[s3 & 255]) ^ kr[2],
            ((S[s3 >> 24] << 24) | (S[(s2 >> 16) & 255] << 16) | (S[(s1 >> 8) & 255] << 8) | S[s0 & 255]) ^ kr[3],
        )

    def encrypt_block(self, b: bytes) -> bytes:
        w = [int.from_bytes(b[4*i:4*i+4], "big") for i in range(4)]
        return b"".join(x.to_bytes(4, "big") for x in self.encrypt_words(*w))

    def decrypt_block(self, b: bytes) -> bytes:
        w = [int.from_bytes(b[4*i:4*i+4], "big") for i in range(4)]
        return b"".join(x.to_bytes(4, "big") for x in self.decrypt_words(*w))


# Validasi resmi FIPS-197 Appendix C.1 + round-trip 1000 blok acak dengan S-box K7
key_fips = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
pt_fips = bytes.fromhex("00112233445566778899aabbccddeeff")
ct_fips = "69c4e0d86a7b0430d8cdb78070b4c55a"
aes_std = AES128(key_fips, S_AES)
assert aes_std.encrypt_block(pt_fips).hex() == ct_fips
assert aes_std.decrypt_block(bytes.fromhex(ct_fips)) == pt_fips

aes_k7_test = AES128(key_fips, S_K7)
rng_rt = np.random.default_rng(SEED)
for blk in rng_rt.integers(0, 256, size=(1000, 16), dtype=np.uint8):
    b_raw = blk.tobytes()
    assert aes_k7_test.decrypt_block(aes_k7_test.encrypt_block(b_raw)) == b_raw

print("✔ AES-128 lolos vektor uji resmi FIPS-197 Appendix C.1:", ct_fips)
print("✔ S-box K7 ciphertext FIPS plaintext:", aes_k7_test.encrypt_block(pt_fips).hex())
print("✔ Round-trip 1.000/1.000 blok acak dengan S-box K7 kembali identik 100%")

## 4.1 Enkripsi & Dekripsi Citra Mandrill (CBC Dua Arah *Domain-Separated IV* vs ECB)
Kunci baku terdokumentasi untuk reproduktibilitas: `KRIPTOGRAFI-KLP7` (16 byte) dan IV `000102…0f`.

In [ ]:
KEY = b"KRIPTOGRAFI-KLP7"
IV = bytes.fromhex("000102030405060708090a0b0c0d0e0f")


def _to_words(data: bytes):
    return np.frombuffer(data, dtype=">u4").astype(np.int64).reshape(-1, 4).tolist()


def _from_words(words) -> bytes:
    return np.array(words, dtype=">u4").tobytes()


def _cbc_enc(aes, blocks, ivw):
    p0, p1, p2, p3 = ivw; out = []; enc = aes.encrypt_words
    for b in blocks:
        p0, p1, p2, p3 = enc(b[0] ^ p0, b[1] ^ p1, b[2] ^ p2, b[3] ^ p3)
        out.append((p0, p1, p2, p3))
    return out


def _cbc_dec(aes, blocks, ivw):
    prev = ivw; out = []; dec = aes.decrypt_words
    for b in blocks:
        d = dec(*b)
        out.append((d[0] ^ prev[0], d[1] ^ prev[1], d[2] ^ prev[2], d[3] ^ prev[3]))
        prev = b
    return out


def encrypt_image(im: np.ndarray, key: bytes, iv: bytes, sbox, mode: str = "CBC Dua Arah"):
    t0 = time.perf_counter()
    flat = im.astype(np.uint8).tobytes()
    pad = (-len(flat)) % 16
    aes = AES128(key, sbox)
    blocks = _to_words(flat + b"\x00" * pad)
    if mode == "ECB":
        c = [aes.encrypt_words(*b) for b in blocks]
    else:
        iv_fwd = _to_words(iv)[0]
        c = _cbc_enc(aes, blocks, iv_fwd)
        if mode == "CBC Dua Arah":
            iv_rev = _to_words(bytes(b ^ 0xA5 for b in iv))[0]
            c = _cbc_enc(aes, c[::-1], iv_rev)[::-1]
    c_flat = np.frombuffer(_from_words(c), dtype=np.uint8).copy()
    n = int(np.prod(im.shape))
    return {"cipher_flat": c_flat, "cipher_img": c_flat[:n].reshape(im.shape), "pad": pad, "elapsed": time.perf_counter() - t0}


def decrypt_image(c_flat: np.ndarray, shape: tuple, key: bytes, iv: bytes, sbox, mode: str = "CBC Dua Arah"):
    t0 = time.perf_counter()
    aes = AES128(key, sbox)
    blocks = _to_words(c_flat.astype(np.uint8).tobytes())
    if mode == "ECB":
        p = [aes.decrypt_words(*b) for b in blocks]
    else:
        if mode == "CBC Dua Arah":
            iv_rev = _to_words(bytes(b ^ 0xA5 for b in iv))[0]
            blocks = _cbc_dec(aes, blocks[::-1], iv_rev)[::-1]
        iv_fwd = _to_words(iv)[0]
        p = _cbc_dec(aes, blocks, iv_fwd)
    n = int(np.prod(shape))
    dec = np.frombuffer(_from_words(p)[:n], dtype=np.uint8).reshape(shape).copy()
    return {"dec_img": dec, "elapsed": time.perf_counter() - t0}


# Uji padding aman pada citra ukuran bukan kelipatan 16 byte (101x101x3)
odd_test = img[:101, :101].copy()
e_odd = encrypt_image(odd_test, KEY, IV, S_K7)
d_odd = decrypt_image(e_odd["cipher_flat"], odd_test.shape, KEY, IV, S_K7)
assert np.array_equal(odd_test, d_odd["dec_img"])

# Enkripsi & dekripsi penuh Mandrill 512x512x3
enc_k7 = encrypt_image(img, KEY, IV, S_K7, mode="CBC Dua Arah")
dec_k7 = decrypt_image(enc_k7["cipher_flat"], img.shape, KEY, IV, S_K7, mode="CBC Dua Arah")
enc_aes = encrypt_image(img, KEY, IV, S_AES, mode="CBC Dua Arah")
enc_ecb = encrypt_image(img, KEY, IV, S_K7, mode="ECB")

cipher_k7, dekripsi_k7 = enc_k7["cipher_img"], dec_k7["dec_img"]
cipher_aes, cipher_ecb = enc_aes["cipher_img"], enc_ecb["cipher_img"]

assert np.array_equal(img, dekripsi_k7), "Dekripsi tidak identik!"
Image.fromarray(cipher_k7).save("mandrill_encrypted_k7.png")
Image.fromarray(dekripsi_k7).save("mandrill_decrypted_k7.png")

print(f"✔ Uji ukuran non-kelipatan-16 (101×101×3, pad={e_odd['pad']}) lossless: YA")
print(f"✔ Mandrill 512×512×3 — Enkripsi: {enc_k7['elapsed']:.2f} dtk | Dekripsi: {dec_k7['elapsed']:.2f} dtk | Piksel beda: 0")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, im_show, ttl in zip(
    axes,
    [img, cipher_k7, dekripsi_k7, cipher_ecb],
    ["Citra Asli (Plain)", "Cipher · S-box K7\n(CBC Dua Arah)", "Hasil Dekripsi\n(100% Identik)", "Pembanding ECB\n(S-box K7)"],
):
    ax.imshow(im_show); ax.set_title(ttl, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.savefig("enkripsi_mandrill_kelompok7.png", dpi=150); plt.show()

## 4.2 Analisis Histogram & Uji Keseragaman Chi-Square ($\chi^2$)
Untuk citra $512 \times 512$, frekuensi ideal tiap intensitas (0–255) adalah $1.024$. Histogram dinyatakan seragam (*uniform*) pada $\alpha = 0{,}05$ ($df=255$) jika $\chi^2 < 293{,}2478$.

In [ ]:
def chi_square(ch: np.ndarray) -> float:
    o = np.bincount(ch.ravel(), minlength=256)
    e = ch.size / 256.0
    return float(((o - e) ** 2 / e).sum())


fig, axes = plt.subplots(2, 3, figsize=(13.5, 5.8), sharex=True)
for c, (nm, col) in enumerate(zip(["Red", "Green", "Blue"], ["#dc2626", "#16a34a", "#2563eb"])):
    for row_i, (arr, tag) in enumerate([(img, "Plain"), (cipher_k7, "Cipher S-box K7")]):
        h = np.bincount(arr[..., c].ravel(), minlength=256)
        axes[row_i, c].bar(np.arange(256), h, width=1.0, color=col)
        axes[row_i, c].set_title(f"{tag} — Kanal {nm}", fontsize=10); axes[row_i, c].set_xlim(0, 255)
    axes[1, c].axhline(1024, color="k", ls="--", lw=1, label="Ideal = 1024")
    axes[1, c].set_ylim(0, 1350); axes[1, c].legend(fontsize=8, loc="lower right"); axes[1, c].set_xlabel("Intensitas")
axes[0, 0].set_ylabel("Frekuensi"); axes[1, 0].set_ylabel("Frekuensi")
plt.suptitle("Histogram Citra Mandrill: Plain vs Cipher (S-box K7)", fontsize=11.5)
plt.tight_layout(); plt.savefig("histogram_kelompok7.png", dpi=150); plt.show()

rows_chi = []
for lbl, arr in [("Plain", img), ("Cipher S-box K7", cipher_k7), ("Cipher AES Standar", cipher_aes)]:
    for c, nm in enumerate(["Red", "Green", "Blue"]):
        h = np.bincount(arr[..., c].ravel(), minlength=256)
        x2 = chi_square(arr[..., c])
        rows_chi.append([lbl, nm, f"{int((h > 0).sum())}/256", int(h.min()), int(h.max()), round(x2, 2),
                         "Lolos (Uniform)" if x2 < CHI2_CRIT else "Tidak lolos"])
df_chi = pd.DataFrame(rows_chi, columns=["Citra", "Kanal", "Intensitas", "Min", "Maks", "Chi-square", f"Uji χ² (< {CHI2_CRIT})"])
display(df_chi)

## 4.3 Analisis Entropy & Korelasi Piksel Bertetangga (Horizontal, Vertikal, Diagonal)
Berbeda dengan pengujian parsial, korelasi dihitung secara **simetris** untuk ketiga arah (**H, V, D**) baik pada citra **Plain** maupun **Cipher**, disertai *scatter plot* sebaran pasangan piksel bertetangga.

In [ ]:
def entropy(ch: np.ndarray) -> float:
    p = np.bincount(ch.ravel(), minlength=256) / ch.size
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def correlation_pairs(ch: np.ndarray, direction: str, n: int = 3000, seed: int = SEED):
    rng = np.random.default_rng(seed)
    h, w = ch.shape
    if direction == "horizontal":
        y, x = rng.integers(0, h, n), rng.integers(0, w - 1, n); a, b = ch[y, x], ch[y, x + 1]
    elif direction == "vertical":
        y, x = rng.integers(0, h - 1, n), rng.integers(0, w, n); a, b = ch[y, x], ch[y + 1, x]
    else:
        y, x = rng.integers(0, h - 1, n), rng.integers(0, w - 1, n); a, b = ch[y, x], ch[y + 1, x + 1]
    return a.astype(np.float64), b.astype(np.float64)


def correlation(ch: np.ndarray, direction: str) -> float:
    a, b = correlation_pairs(ch, direction)
    return float(np.corrcoef(a, b)[0, 1])


rows_ec = []
for c, nm in enumerate(["Red", "Green", "Blue"]):
    rows_ec.append({
        "Kanal": nm,
        "Entropy Plain": entropy(img[..., c]),
        "Entropy Cipher K7": entropy(cipher_k7[..., c]),
        "Entropy Cipher AES": entropy(cipher_aes[..., c]),
        "Korelasi H Plain": correlation(img[..., c], "horizontal"),
        "Korelasi H Cipher K7": correlation(cipher_k7[..., c], "horizontal"),
        "Korelasi V Plain": correlation(img[..., c], "vertical"),
        "Korelasi V Cipher K7": correlation(cipher_k7[..., c], "vertical"),
        "Korelasi D Plain": correlation(img[..., c], "diagonal"),
        "Korelasi D Cipher K7": correlation(cipher_k7[..., c], "diagonal"),
    })
df_ec = pd.DataFrame(rows_ec)
display(df_ec.round(5))
print(f"Rata-rata Entropy Cipher S-box K7 : {df_ec['Entropy Cipher K7'].mean():.5f} bit (ideal 8.00000)")
print(f"Rata-rata Entropy Cipher AES      : {df_ec['Entropy Cipher AES'].mean():.5f} bit")

fig, axes = plt.subplots(2, 3, figsize=(12.0, 5.8))
for col_i, (dkey, dlabel) in enumerate([("horizontal", "Horizontal"), ("vertical", "Vertikal"), ("diagonal", "Diagonal")]):
    ap, bp = correlation_pairs(img[..., 0], dkey, n=2000)
    ac_, bc_ = correlation_pairs(cipher_k7[..., 0], dkey, n=2000)
    axes[0, col_i].scatter(ap, bp, s=2, alpha=0.35, color="#0f766e")
    axes[0, col_i].set_title(f"Plain (R) · {dlabel}"); axes[0, col_i].set_xlim(0, 255); axes[0, col_i].set_ylim(0, 255)
    axes[1, col_i].scatter(ac_, bc_, s=2, alpha=0.35, color="#2563eb")
    axes[1, col_i].set_title(f"Cipher K7 (R) · {dlabel}"); axes[1, col_i].set_xlim(0, 255); axes[1, col_i].set_ylim(0, 255)
axes[0, 0].set_ylabel("Piksel tetangga"); axes[1, 0].set_ylabel("Piksel tetangga")
for ax in axes[1]:
    ax.set_xlabel("Piksel acuan")
plt.suptitle("Korelasi Piksel Bertetangga: Plain vs Cipher (S-box K7)", fontsize=11)
plt.tight_layout(); plt.savefig("korelasi_kelompok7.png", dpi=150); plt.show()

## 4.4 Analisis Serangan Diferensial: NPCR, UACI (Uji Wu dkk., 2011), Sensitivitas Kunci 1-Bit, & Pembanding ECB
Untuk citra $512 \times 512$ pada $\alpha = 0{,}05$, ambang lolos uji **Wu, Noonan & Agaian (2011)** adalah:
- **NPCR** $> 99{,}5893\%$ (ideal $99{,}6094\%$)
- **UACI** di antara $33{,}3730\%$ dan $33{,}5541\%$ (ideal $33{,}4635\%$)

In [ ]:
NPCR_CRIT, UACI_LO, UACI_HI = 99.5893, 33.3730, 33.5541


def npcr_uaci(c1: np.ndarray, c2: np.ndarray):
    a, b = c1.astype(np.int64), c2.astype(np.int64)
    return float((a != b).mean() * 100.0), float((np.abs(a - b) / 255.0).mean() * 100.0)


positions = [(0, 0, 0), (128, 384, 1), (256, 256, 1), (384, 128, 2), (511, 511, 2)]
rows_wu, n_pass_k7, u_pass_k7 = [], 0, 0
k7_n_all, k7_u_all, aes_n_all, aes_u_all = [], [], [], []

for py, px, ch in positions:
    im2 = img.copy()
    im2[py, px, ch] = (int(im2[py, px, ch]) + 1) % 256
    c2_k7 = encrypt_image(im2, KEY, IV, S_K7)["cipher_img"]
    c2_aes = encrypt_image(im2, KEY, IV, S_AES)["cipher_img"]
    nk, uk = zip(*[npcr_uaci(cipher_k7[..., k], c2_k7[..., k]) for k in range(3)])
    na, ua = zip(*[npcr_uaci(cipher_aes[..., k], c2_aes[..., k]) for k in range(3)])
    n_pass_k7 += sum(v > NPCR_CRIT for v in nk)
    u_pass_k7 += sum(UACI_LO < v < UACI_HI for v in uk)
    k7_n_all.append(np.mean(nk)); k7_u_all.append(np.mean(uk))
    aes_n_all.append(np.mean(na)); aes_u_all.append(np.mean(ua))
    rows_wu.append([f"({py}, {px}) {'RGB'[ch]}", *np.round(nk, 4), round(np.mean(nk), 4), *np.round(uk, 4), round(np.mean(uk), 4)])

df_wu = pd.DataFrame(rows_wu, columns=["Posisi Ubah 1 Piksel", "NPCR R", "NPCR G", "NPCR B", "NPCR Rata2", "UACI R", "UACI G", "UACI B", "UACI Rata2"])
display(df_wu)

# Sensitivitas kunci 1 bit & pembanding ECB
KEY2 = KEY[:-1] + bytes([KEY[-1] ^ 1])
c_key2 = encrypt_image(img, KEY2, IV, S_K7)["cipher_img"]
kn, ku = zip(*[npcr_uaci(cipher_k7[..., k], c_key2[..., k]) for k in range(3)])

im_mid = img.copy(); im_mid[256, 256, 1] = (int(im_mid[256, 256, 1]) + 1) % 256
c_ecb2 = encrypt_image(im_mid, KEY, IV, S_K7, mode="ECB")["cipher_img"]
en, eu = zip(*[npcr_uaci(cipher_ecb[..., k], c_ecb2[..., k]) for k in range(3)])

df_sken = pd.DataFrame([
    ["CBC Dua Arah · Ubah 1 piksel (rata2 5 posisi) — S-box K7", np.mean(k7_n_all), np.mean(k7_u_all)],
    ["CBC Dua Arah · Ubah 1 piksel (rata2 5 posisi) — AES Standar", np.mean(aes_n_all), np.mean(aes_u_all)],
    ["Sensitivitas Kunci (beda 1 bit) — S-box K7", np.mean(kn), np.mean(ku)],
    ["Pembanding Mode ECB (ubah 1 piksel) — S-box K7", np.mean(en), np.mean(eu)],
], columns=["Skenario Pengujian", "NPCR Rata2 (%)", "UACI Rata2 (%)"])
display(df_sken.round(4))
df_sken.to_csv("rekap_enkripsi_kelompok7.csv", index=False)

print(f"Ringkasan Wu dkk. (5 posisi × 3 kanal) S-box K7 : NPCR lolos {n_pass_k7}/15 | UACI lolos {u_pass_k7}/15")
print(f"Rata-rata 5 posisi S-box K7  → NPCR = {np.mean(k7_n_all):.4f}% | UACI = {np.mean(k7_u_all):.4f}%")
print(f"Rata-rata 5 posisi AES Std   → NPCR = {np.mean(aes_n_all):.4f}% | UACI = {np.mean(aes_u_all):.4f}%")

---
# Kesimpulan Akhir — Kelompok 7 (Ilmu Komputer UNNES 2026)

1. **Eksplorasi 30 Polinomial & Ablasi ML Surrogate (Cahya Din Ramadhan):** Seluruh 30 polinomial irreducible derajat 8 di $\text{GF}(2^8)$ berhasil diskrining, menempatkan `0x17B` ($x^8+x^6+x^5+x^4+x^3+x+1$) di peringkat #1. Studi ablasi membuktikan `KNN` & `Random Forest` pada 64-bit mentah gagal ($R^2 \approx 0$), `Ridge 1-baris` hanya mampu memprediksi SAC namun gagal di BIC-SAC ($R^2 < 0$), sedangkan **Ridge Komposit (`Row + Pairwise-XOR`)** memprediksi SAC dan BIC-SAC dengan $R^2 = 1{,}00000$.
2. **Optimasi SAGA & Konstanta $C$ (Muhammad Nabil Fikri):** *Surrogate-Assisted Genetic Algorithm* menemukan matriks affine non-sirkulan `A_K7` (`rows = [6, 47, 61, 82, 110, 187, 216, 218]`) yang secara simultan mencapai **SAC = `0,50000`** (deviasi `0,01562`) dan **BIC-SAC = `0,50000`** (deviasi `0,00446`). Optimasi konstanta $C = \texttt{0xEE}$ menghasilkan **0 Fixed Point, 0 Opposite Fixed Point, dan 1 siklus tunggal penuh `[256]`**.
3. **Keunggulan Komparatif S-box (Fathan Nabil Rahman):** `S-box K7` memenuhi seluruh kriteria wajib (**Balance**, **Bijective**, **NL = 112**, **BIC-NL = 112**, **LAP bias = 0,0625**, **DAP = 0,015625**) dan **mengungguli AES standar, Alamsyah (2023), maupun Alamsyah (2025) `S-box₄₄`** pada Deviasi SAC (`0,01562` vs `0,02515`), Deviasi BIC-SAC (`0,00446` vs `0,00949`), Skor $S = 0{,}00000$, Skor $SV = 16{,}00000$, serta bebas *fixed point* & *opposite fixed point*.
4. **Implementasi Enkripsi Citra AES-128 (Rashya Farellyndo Revanza):** AES-128 T-table tervalidasi resmi **FIPS-197 Appendix C.1** dengan dekripsi 100% *lossless*. Enkripsi Mandrill dengan mode **CBC Dua Arah (*domain-separated IV*)** menghasilkan histogram seragam lolos uji $\chi^2 < 293{,}25$ di ketiga kanal RGB, **Entropy ≈ 7,9993 bit**, korelasi piksel bertetangga $\approx 0$ di arah H/V/D, serta **NPCR ≈ 99,61%** dan **UACI ≈ 33,45%** yang memenuhi kriteria diferensial Wu dkk. (2011) dan sensitif terhadap perubahan 1 bit kunci.